# What the models use: variable contributions and state dynamics

Three figures on Cam-CAN HADS. Each has one panel per task (rest, movie) × phenotype (anxiety, depression).

1. **Every saved model.** `select-bstm` refits and saves its best state sets per (model, arm)
   (`models/*.joblib`). For each one, the share of the model's contribution that comes from
   the covariates, the transition cells, the occupancy and the dynamics. **1.2** shows the
   covariates and the eight dynamics one by one: the only variables that mean the same thing
   in every state set.
2. **One model, in depth.** `CHOICE` picks one (model, arm, state set), by default the best
   across the four panels. It is refit in every panel. A SHAP beeswarm shows its top variables:
   which way each one pushes the prediction, and for which subjects.
3. **The state dynamics behind it.** For figure 2's state set, the Normal group's average
   state diagram, and beside it how the `CASE_FROM`+ group differs.

**Contribution = out-of-fold SHAP**, in units of the predicted HADS level, the same for ridge
and hgb. Each fold's model only explains subjects it did not see, so this is what the model
uses *and generalises with*. The folds are `select-bstm`'s own (`KFold(5)`, `SEED`). For ridge,
SHAP is exact and needs no library: coefficient × standardised value. For hgb it is
`shap.TreeExplainer`, also exact. Needs `shap` (`pip install --user shap`).

The saved `.joblib` files only say *which* models to explain. Every fold is refit here, so
the in-sample fit stored in them is never used.

In [ ]:
import os
from pathlib import Path

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
CAMCAN = Path(os.environ.get("CAMCAN_PHENO",
                             "/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
                             "opendata/paule_toussaint_camcan01870"))
PHENO_FILES = [(CAMCAN / "approved_data.tsv", "\t"),    # HADS
               (CAMCAN / "standard_data.csv", ",")]      # Age, Sex
PHENO_ID = "CCID"

TARGETS = {"anxiety":    "additional_HADS_anx_category",
           "depression": "additional_HADS_dep_category"}
ORDINAL_LEVELS = ["Normal", "Mild", "Moderate", "Severe"]   # low -> high
TASKS = {"rest": "rest", "movie": "movie"}                  # matched case-insensitively
COHORTS = ["camcan", "camcan_rest"]
COVARIATES = ["Age", "Sex", "n_transitions"]
CATEGORICAL = ["Sex"]

# Figure 1: which saved models. None = every ridge / hgb / lgbm model saved
# (logistic is a classifier, another scale).
MODELS = None
SEED = 0                                         # fold seed for the out-of-fold SHAP
# Figure 1's SHAP (minutes) is stored here and reused while the saved models and SEED are
# unchanged, so re-drawing costs seconds. Outside outputs/, git-ignored. None = always recompute.
CACHE_DIR = Path("cache")
N_JOBS = (len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity")
          else os.cpu_count())                   # cores (your SLURM share)
REPO = Path.cwd().resolve().parent               # for fmri_decomposition (make_model)

# Figures 2 and 3: the deep dive. None = the best (model, arm, state set) by mean CV score
# over the four panels. Or name one, e.g.
# CHOICE = dict(model="hgb", arm="all [cond]", atlas="yeo7", window_s=30, states="HMM1_pca3_8")
CHOICE = None
TOP_VARS = 15                                    # variables in each beeswarm
CASE_FROM = "Mild"                               # figure 3: this level and above vs Normal
EDGE_TOP = 12                                    # figure 3: the Normal group's strongest arrows (+ every change)
ALPHA = 0.10                                     # figure 3: BH-FDR on the edge differences
SAVE_DIR = None                                  # e.g. Path("figures/09") to save PNG + SVG

# used only by the loaders shared with 08.1
MODEL, ARM, TOP_N = "ridge", {"bstm": "cells [joint]"}, 10

In [ ]:
import sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats

INK, INK2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#a3a29d", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "axes.titlesize": 11,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "font.size": 9, "legend.frameon": False})

def hive(path, key):
    """One `key=value` partition from a path, by name."""
    return next((p.split("=", 1)[1] for p in Path(path).parts if p.startswith(f"{key}=")), None)

def fmt_window(w):
    return str(int(w)) if float(w).is_integer() else str(w)

def set_label(r):
    """A readable name for one summary row: state set or static-FC atlas."""
    if r.get("tree") == "fcm":
        return r["atlas"]
    w = "1 TR" if float(r["window_s"]) < 0 else f"{fmt_window(r['window_s'])}s"
    return f"{r['states']} · {r['atlas']} · {w}"

def save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        for ext in ("png", "svg"):
            fig.savefig(Path(SAVE_DIR) / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def design(cov, intercept=True):
    """Covariates -> numeric matrix: categoricals one-hot, gaps mean-filled."""
    X = pd.get_dummies(cov, columns=[c for c in CATEGORICAL if c in cov],
                       drop_first=True).astype(float)
    X = X.fillna(X.mean()).to_numpy()
    return np.column_stack([np.ones(len(X)), X]) if intercept else X
import matplotlib

BLOCK_COLOUR = {"covariates": "#a3a29d", "cells": "#2a78d6", "occupancy": "#eb6834", "dynamics": "#1baf7a"}
MODEL_COLOUR = {"ridge": "#2a78d6", "hgb": "#eb6834", "lgbm": "#1baf7a"}
UP, DOWN = "#e34948", "#2a78d6"                  # higher / lower in the symptom group
VALUE_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list("value", [DOWN, "#c3c2bd", UP])

In [ ]:
def read_summaries(tree):
    files = sorted((ROOT / f"{tree}_selection").glob("task=*/target=*/summary.csv"))
    if not files:
        raise SystemExit(f"no summary.csv under {ROOT / f'{tree}_selection'}")
    return pd.concat([pd.read_csv(p).assign(tree=tree, task=hive(p, "task"),
                                            target=hive(p, "target"))
                      for p in files], ignore_index=True)

def read_pheno(covariates):
    frames = [pd.read_csv(p, sep=s, dtype={PHENO_ID: str}) for p, s in PHENO_FILES]
    ph = frames[0]
    for f in frames[1:]:
        ph = ph.merge(f.drop(columns=[c for c in f if c in ph and c != PHENO_ID]),
                      on=PHENO_ID, how="outer")
    lower = {c.lower(): c for c in ph.columns}              # Age vs age
    ph = ph.rename(columns={lower[c.lower()]: c for c in covariates if c.lower() in lower})
    ph["sub"] = ph[PHENO_ID].astype(str).str.strip().str.upper()
    code = {lvl: k for k, lvl in enumerate(ORDINAL_LEVELS)}
    for lab, col in TARGETS.items():
        ph[lab] = ph[col].astype("string").str.strip().map(code).astype(float)
    for c in set(covariates) - set(CATEGORICAL) - {"n_transitions"}:
        ph[c] = pd.to_numeric(ph[c], errors="coerce")
    return ph[["sub", *TARGETS, *(c for c in covariates if c in ph)]]

def load_set(atlas, window_s, states, task):
    """One state set's transition table for one task, one row per subject."""
    base = (ROOT / "transitions" / f"atlas={atlas}" / f"window_s={fmt_window(window_s)}"
            / f"states={states}")
    files = [p for p in base.rglob("subjects.parquet")
             if hive(p, "cohort") in COHORTS
             and (hive(p, "task") or task).lower() == TASKS[task].lower()]
    if not files:
        raise SystemExit(f"no transition table for {base} task={task}")
    d = pd.concat(map(pd.read_parquet, files), ignore_index=True)
    d = d[d["task"].astype(str).str.lower() == TASKS[task].lower()]
    d["sub"] = d["sub"].astype(str).str.strip().str.upper()
    if d["sub"].duplicated().any():
        raise SystemExit(f"{states} {task}: a subject appears twice -- narrow COHORTS")
    return d

def cell_cols(d):
    return [c for c in d.columns if "->" in c]

def as_p_cond(d, K):
    """p_joint -> Markov p(j | i): back to counts, then renormalise each row."""
    cols = cell_cols(d)
    M = d[cols].to_numpy(float).reshape(len(d), K, K) * d["n_transitions"].to_numpy(float)[:, None, None]
    s = M.sum(2, keepdims=True)
    return pd.DataFrame(np.divide(M, s, out=np.zeros_like(M), where=s > 0).reshape(len(d), -1),
                        columns=cols, index=d.index)

def partial_spearman(A, y, cov):
    """Rank everything, residualise on the covariates, correlate. Returns r, p."""
    X = design(cov)
    R = np.column_stack([stats.rankdata(y), np.apply_along_axis(stats.rankdata, 0, A)])
    R = R - X @ np.linalg.lstsq(X, R, rcond=None)[0]
    R = (R - R.mean(0)) / np.where(R.std(0) > 0, R.std(0), np.nan)
    r = R[:, 1:].T @ R[:, 0] / len(R)
    df = len(R) - X.shape[1] - 1
    p = 2 * stats.t.sf(np.abs(r * np.sqrt(df / (1 - r ** 2))), df)
    return r, p

def top_sets(task, lab, n=TOP_N, arm=None):
    """The n best bstm state sets of one ranking (MODEL, `arm` or ARM), as summary.csv rows."""
    g = summ[(summ["tree"] == "bstm") & (summ["task"] == task) & (summ["phenotype"] == lab)
             & (summ["arm"] == (arm or ARM["bstm"])) & (summ["model"] == MODEL)]
    return (g.sort_values("mean", ascending=False)
             .drop_duplicates(["atlas", "window_s", "states"]).head(n).reset_index(drop=True))

# What each select-bstm arm hands the model, as in bstm_selection.build.
DYNAMICS = ["n_states_visited", "n_distinct_transitions", "switch_rate", "self_transition_rate",
            "switches_per_min", "mean_dwell_s", "entropy_rate_bits", "dispersion"]
ARM_BLOCKS = {"cells": ["cells"], "occupancy": ["occupancy"], "dynamics": ["dynamics"],
              "summary": ["occupancy", "dynamics"], "all": ["cells", "occupancy", "dynamics"]}

def arm_columns(t, arm):
    """{block: its columns in table t} for one arm, e.g. "all [cond]" -> cells, occupancy, dynamics."""
    cols = {"cells": cell_cols(t), "occupancy": [c for c in t.columns if c.startswith("occ_")],
            "dynamics": [c for c in DYNAMICS if c in t.columns]}
    return {b: cols[b] for b in ARM_BLOCKS[arm.split(" [")[0]]}

def merged(s, task, lab, arm=None):
    """One state set's table joined to phenotype, cells as the arm asks for ([cond] -> p(j | i))."""
    t = load_set(s["atlas"], s["window_s"], s["states"], task)
    if "[cond]" in (arm or ARM["bstm"]):
        t[cell_cols(t)] = as_p_cond(t, int(s["K"]))
    d = t.merge(pheno, on="sub", suffixes=("", "_pheno")).dropna(
        subset=[lab, *(c for c in COVARIATES if c not in CATEGORICAL)])
    return d, cell_cols(t)

label_of = {col: lab for lab, col in TARGETS.items()}
summ = pd.concat([read_summaries("bstm"), read_summaries("fcm")], ignore_index=True)
summ = summ[summ["target"].isin(label_of) & summ["task"].isin(TASKS)].assign(phenotype=lambda d: d["target"].map(label_of))
summ["label"] = summ.apply(set_label, axis=1)

import hashlib, joblib, time
from joblib import Parallel, delayed

pheno = read_pheno(COVARIATES)

def saved_models():
    """Every model select-bstm saved, as one row of metadata each."""
    rows = []
    for p in sorted((ROOT / "bstm_selection").glob("task=*/target=*/models/*.joblib")):
        m = joblib.load(p)
        rows.append({"task": hive(p, "task"), "target": hive(p, "target"), "model": m["model"],
                     "arm": m["features"] + ("" if m["p_norm"] in ("-", None) else f" [{m['p_norm']}]"),
                     "atlas": m["atlas"], "window_s": float(m["window_s"]), "states": m["states"],
                     "covariates": tuple(m["covariates"]), "cv": m["mean_cv_score"]})
    d = pd.DataFrame(rows)
    d = d[d["task"].isin(TASKS) & d["target"].isin(label_of) & (d["model"] != "logistic")
          & (d["model"].isin(MODELS) if MODELS else True)].reset_index(drop=True)
    d["phenotype"] = d["target"].map(label_of)
    d["label"] = d["model"] + " · " + d["arm"] + " · " + d.apply(set_label, axis=1)
    return d

saved = saved_models()
saved.pivot_table(index=["task", "phenotype"], columns="model", values="cv", aggfunc="size")

In [ ]:
def matrix(cfg):
    """One config -> (X, y, names, blocks): covariates first, then the arm's variables,
    built as select-bstm builds them."""
    s = {"atlas": cfg["atlas"], "window_s": cfg["window_s"], "states": cfg["states"],
         "K": int(cfg["states"].rsplit("_", 1)[1])}
    d, _ = merged(s, cfg["task"], cfg["phenotype"], cfg["arm"])
    cov = pd.get_dummies(d[list(cfg["covariates"])], drop_first=True).astype(float)
    cov = cov.fillna(cov.mean())
    cols = arm_columns(d, cfg["arm"])
    names = list(cov.columns) + [c for cs in cols.values() for c in cs]
    blocks = ["covariates"] * cov.shape[1] + [b for b, cs in cols.items() for _ in cs]
    X = np.column_stack([cov.to_numpy(), d[names[cov.shape[1]:]].to_numpy(float)])
    return X, d[cfg["phenotype"]].to_numpy(float), names, blocks

def oof_shap(repo, X, y, model, seed):
    """Out-of-fold SHAP on one core: each fold's model explains only the subjects it did not see."""
    import sys
    sys.path.insert(0, repo)
    import numpy as np
    from sklearn.base import clone
    from sklearn.model_selection import KFold
    from threadpoolctl import threadpool_limits
    from fmri_decomposition.bstm_selection import make_model
    est = make_model(model, seed)[0]
    S = np.zeros_like(X, dtype=float)
    with threadpool_limits(1):
        for tr, te in KFold(5, shuffle=True, random_state=seed).split(X):
            m = clone(est).fit(X[tr], y[tr])
            if model == "ridge":          # linear: exact SHAP = coefficient x standardised value
                S[te] = m[-1].coef_ * m[0].transform(X[te])
            else:
                import shap
                S[te] = shap.TreeExplainer(m).shap_values(X[te])
    return S

def _mean_abs(repo, X, y, model, seed):
    return np.abs(oof_shap(repo, X, y, model, seed)).mean(0)

def explain_all():
    """mean |SHAP| of every variable of every saved model. Built lazily, so only a few
    matrices are in memory at once; one core per model, all cores busy."""
    names = []
    def jobs():
        for (X, y, n, b), (_, c) in zip((matrix(c) for _, c in saved.iterrows()), saved.iterrows()):
            names.append((n, b))
            yield delayed(_mean_abs)(str(REPO), X, y, c["model"], SEED)
    imp = Parallel(n_jobs=N_JOBS)(jobs())
    out = pd.concat([pd.DataFrame({"variable": n, "block": b, "mean_abs": v}).assign(
                         **saved.loc[i, ["task", "phenotype", "model", "arm", "label", "cv"]].to_dict())
                     for i, ((n, b), v) in enumerate(zip(names, imp))], ignore_index=True)
    out["share"] = out["mean_abs"] / out.groupby(["task", "phenotype", "label"])["mean_abs"].transform("sum")
    return out

# The cache key is the list of saved models plus the seed: a new selection run invalidates it.
key = hashlib.md5((saved[["task", "target", "model", "arm", "atlas", "window_s", "states"]]
                   .to_csv(index=False) + f"seed={SEED}").encode()).hexdigest()[:12]
cache = CACHE_DIR / f"09_contrib_{key}.parquet" if CACHE_DIR else None
n_hgb = int((saved["model"] != "ridge").sum())
if cache and cache.exists():
    contrib = pd.read_parquet(cache)
    print(f"read {cache} ({len(saved)} models)")
else:
    print(f"explaining {len(saved)} saved models ({n_hgb} boosted) on {N_JOBS} core(s): "
          f"~{n_hgb * 12 / 60 / N_JOBS:.0f}-{n_hgb * 21 / 60 / N_JOBS:.0f} min at most")
    t0 = time.time()
    contrib = explain_all()
    print(f"done in {time.time() - t0:.0f}s")
    if cache:
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        contrib.to_parquet(cache)

## 1. Where each saved model's contribution comes from

One bar per saved model, best CV score on top. The segments are the share of the model's total
mean |SHAP| that comes from each variable type. A model whose bar is mostly grey is mostly
predicting from the covariates. The cells block sums many small contributions (K² of them), so
compare it with care against the eight dynamics.

In [ ]:
ORDER = ["covariates", "cells", "occupancy", "dynamics"]

def plot_shares(ax, c):
    share = c.pivot_table(index="label", columns="block", values="share", aggfunc="sum").reindex(columns=ORDER).fillna(0)
    # grouped by model (ridge, then hgb, ...), best first within each; barh draws bottom-up
    first = c.groupby("label")[["model", "cv"]].first().sort_values(["model", "cv"], ascending=[True, False])
    share = share.loc[first.index[::-1]]
    left = np.zeros(len(share))
    for b in ORDER:
        ax.barh(range(len(share)), share[b], left=left, height=.75, color=BLOCK_COLOUR[b],
                edgecolor=SURFACE, lw=1, label=b)
        left += share[b].to_numpy()
    cv = c.groupby("label")["cv"].first()
    ax.set_yticks(range(len(share)), [f"{l}   ρ={cv[l]:.2f}" for l in share.index], fontsize=6.5)
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))
    models_bottom_up = first["model"].to_numpy()[::-1]
    for k in np.nonzero(models_bottom_up[1:] != models_bottom_up[:-1])[0]:     # between two models
        ax.axhline(k + .5, color=INK, lw=.8)
    for m in dict.fromkeys(models_bottom_up):                                    # the model's name, once
        ax.text(1.01, np.nonzero(models_bottom_up == m)[0].mean(), m, transform=ax.get_yaxis_transform(),
                rotation=-90, va="center", fontsize=8, color=INK2)

n = contrib.groupby(["task", "phenotype"])["label"].nunique().max()
fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False,
                         figsize=(7.5 * len(TARGETS), len(TASKS) * (.9 + .19 * n)))
for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
    plot_shares(axes[i, j], contrib[(contrib["task"] == task) & (contrib["phenotype"] == lab)])
    axes[i, j].set_title(f"{task} · {lab}", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("share of the model's mean |SHAP| (out-of-fold)")
fig.legend(handles=[Patch(color=BLOCK_COLOUR[b], label=b) for b in ORDER], loc="upper right",
           ncol=4, fontsize=8, bbox_to_anchor=(1, 1.0))
fig.suptitle("1  where each saved model's contribution comes from", x=.01, ha="left", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, .97))
save(fig, "fig1_shares")

## 1.2 The covariates and the dynamics, one by one

The only variables that mean the same thing in every state set. One dot per saved model that
contains the variable, coloured by model type; the bar is the median. Dynamics only appear for
models whose arm has them (`dynamics`, `summary`, `all`).

In [ ]:
shared = contrib[contrib["block"].isin(["covariates", "dynamics"])]
VARS = list(dict.fromkeys(shared.loc[shared["block"] == "covariates", "variable"])) + \
       [v for v in DYNAMICS if v in set(shared["variable"])]

fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False, sharey=True,
                         figsize=(7 * len(TARGETS), 3.6 * len(TASKS)))
rng = np.random.default_rng(0)
for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
    ax, g = axes[i, j], shared[(shared["task"] == task) & (shared["phenotype"] == lab)]
    for k, v in enumerate(VARS):
        h = g[g["variable"] == v]
        ax.scatter(k + rng.uniform(-.18, .18, len(h)), h["mean_abs"], s=14, lw=0, alpha=.7,
                   c=h["model"].map(MODEL_COLOUR))
        if len(h):
            ax.plot([k - .28, k + .28], [h["mean_abs"].median()] * 2, color=INK, lw=1.5)
    ax.set_xticks(range(len(VARS)), VARS, rotation=35, ha="right", fontsize=7.5)
    ax.set_title(f"{task} · {lab}", fontsize=10)
for ax in axes[:, 0]:
    ax.set_ylabel("mean |SHAP|  (HADS levels)")
fig.legend(handles=[Line2D([], [], ls="none", marker="o", color=c, label=m) for m, c in MODEL_COLOUR.items()
                    if m in set(shared["model"])] + [Line2D([], [], color=INK, lw=1.5, label="median")],
           loc="upper right", ncol=4, fontsize=8, bbox_to_anchor=(1, 1.0))
fig.suptitle("1.2  covariates and dynamics across the saved models", x=.01, ha="left", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, .96))
save(fig, "fig1_2_shared")

## 2. One model in depth

`CHOICE`, or by default the (model, arm, state set) with the best CV score averaged over the four
panels, among those scored in all four. It is refit and explained in each panel. One row per
variable (the `TOP_VARS` largest mean |SHAP| of that panel), one dot per subject. x is how much that
variable moved the subject's predicted HADS level, out-of-fold. Colour is the subject's value of
the variable, from low (blue) to high (red). Red dots on the right mean "higher values, higher
predicted symptoms".

In [ ]:
def choose():
    if CHOICE:
        return dict(CHOICE, window_s=float(CHOICE["window_s"]))
    b = summ[(summ["tree"] == "bstm") & (summ["arm"] != "(covariates only)")
             & summ["model"].isin(MODELS or ["ridge", "hgb", "lgbm"])]
    m = b.groupby(["model", "arm", "atlas", "window_s", "states"])["mean"].agg(["mean", "size"])
    m = m[m["size"] == len(TASKS) * len(TARGETS)]
    return dict(zip(["model", "arm", "atlas", "window_s", "states"], m["mean"].idxmax()))

pick = choose()
pick["label"] = f"{pick['model']} · {pick['arm']} · {set_label(pick)}"
print("deep dive:", pick["label"])

deep = {}
for task in TASKS:
    for lab in TARGETS:
        cfg = dict(pick, task=task, phenotype=lab, covariates=tuple(COVARIATES))
        X, y, n, b = matrix(cfg)
        deep[task, lab] = dict(X=X, y=y, names=n, blocks=b)
for (key, d), S in zip(deep.items(), Parallel(n_jobs=min(N_JOBS, len(deep)))(
        delayed(oof_shap)(str(REPO), d["X"], d["y"], pick["model"], SEED) for d in deep.values())):
    d["S"] = S

In [ ]:
SHORT = {"covariates": "cov", "cells": "cell", "occupancy": "occ", "dynamics": "dyn"}

def beeswarm(ax, d):
    S, X = d["S"], d["X"]
    top = np.argsort(np.abs(S).mean(0))[::-1][:TOP_VARS][::-1]          # largest on top
    rng = np.random.default_rng(0)
    for row, j in enumerate(top):
        s = S[:, j]
        hist, edges = np.histogram(s, bins=40)
        dens = hist[np.clip(np.digitize(s, edges) - 1, 0, len(hist) - 1)] / max(hist.max(), 1)
        yy = row + rng.uniform(-1, 1, len(s)) * .38 * dens               # wider where dots pile up
        val = pd.Series(X[:, j]).rank(pct=True).to_numpy()
        ax.scatter(s, yy, c=val, cmap=VALUE_CMAP, vmin=0, vmax=1, s=5, lw=0, alpha=.8, rasterized=True)
    ax.set_yticks(range(len(top)), [f"{d['names'][j]}  ({SHORT[d['blocks'][j]]})" for j in top], fontsize=7.5)
    ax.axvline(0, color=MUTED, lw=.8)
    ax.set_ylim(-.7, len(top) - .3)

fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False,
                         figsize=(6.8 * len(TARGETS), (1 + .32 * TOP_VARS) * len(TASKS)))
for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
    beeswarm(axes[i, j], deep[task, lab])
    axes[i, j].set_title(f"{task} · {lab}   (n={len(deep[task, lab]['y'])})", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("SHAP: push on the predicted HADS level (out-of-fold)")
cb = fig.colorbar(matplotlib.cm.ScalarMappable(cmap=VALUE_CMAP), ax=axes, shrink=.35, aspect=25, pad=.02)
cb.set_ticks([0, 1], labels=["low", "high"])
cb.set_label("the variable's value", fontsize=8)
fig.suptitle(f"2  {pick['label']}", x=.01, ha="left", fontsize=10)
save(fig, "fig2_beeswarm")

## 3. The state dynamics of that state set

Figure 2's state set, one row per task and three diagrams per row. **Normal** means Normal on both
HADS scales. **Anxiety** and **depression** mean `CASE_FROM` or above on that scale; someone can be
in both. Each diagram is that group's own dynamics, pooled over its subjects' transitions.

How to read one diagram:

* **Circle area** is the share of time spent in the state. The number under it is that share.
* **The loop** on a circle is p(stay): the thicker it is, the longer the state is held.
* **Arrow width** is p(j | i): of the moves that leave state i, the share that go to state j.
  The same arrows (the Normal group's `EDGE_TOP` strongest, plus any that change) are drawn on
  the same width scale in all three diagrams of a row.
* **On the anxiety and depression diagrams**, the dashed ring is the Normal group's time in
  the state. Anything drawn in the group's colour **differs from Normal**, with the change in
  percentage points beside it. The test regresses each subject's p(j | i), p(stay) and
  occupancy on the group plus `COVARIATES`, with BH-FDR at `ALPHA` over all of them. Everything
  else is grey.

In [ ]:
from matplotlib.patches import Arc, Circle, FancyArrowPatch

CASE = ORDINAL_LEVELS.index(CASE_FROM)
K = int(pick["states"].rsplit("_", 1)[1])
GROUPS = {"Normal": INK2, "anxiety": "#6250d6", "depression": "#eb6834"}   # identity, not direction
GREY, NODE = "#a3a29d", "#ecebe7"

def groups_of(task):
    """Pooled dynamics of Normal / anxiety / depression for one task, and each group's
    covariate-adjusted test against Normal."""
    t = load_set(pick["atlas"], pick["window_s"], pick["states"], task)
    d = t.merge(pheno, on="sub", suffixes=("", "_pheno")).dropna(
        subset=[*TARGETS, *(c for c in COVARIATES if c not in CATEGORICAL)])
    cells = cell_cols(d)
    member = {"Normal": ((d[list(TARGETS)] == 0).all(axis=1)).to_numpy(),
              **{lab: (d[lab] >= CASE).to_numpy() for lab in TARGETS}}
    counts = (d[cells].to_numpy(float) * d["n_transitions"].to_numpy(float)[:, None]).reshape(-1, K, K)
    occ = d[[f"occ_{k}" for k in range(K)]].to_numpy(float)
    per_subject = np.column_stack([as_p_cond(d, K).to_numpy(float), occ])  # K*K p(j|i), then K occupancies
    out = {}
    for g, m in member.items():
        c = counts[m].sum(0)
        out[g] = dict(T=c / np.where(c.sum(1, keepdims=True) > 0, c.sum(1, keepdims=True), 1),
                      occ=occ[m].mean(0), n=int(m.sum()))
        if g == "Normal":
            continue
        rows = m | member["Normal"]                                       # this group vs Normal only
        X = np.column_stack([m[rows].astype(float), design(d.loc[rows, COVARIATES])])
        Y = per_subject[rows]
        B = np.linalg.lstsq(X, Y, rcond=None)[0]
        dof = len(X) - X.shape[1]
        se = np.sqrt(((Y - X @ B) ** 2).sum(0) / dof * np.linalg.pinv(X.T @ X)[0, 0])
        with np.errstate(divide="ignore", invalid="ignore"):
            p = 2 * stats.t.sf(np.abs(B[0] / se), dof)
        ok = np.isfinite(p)
        q = np.ones_like(p)
        q[ok] = stats.false_discovery_control(p[ok], method="bh")
        out[g].update(sig_T=(q[:K * K] < ALPHA).reshape(K, K), sig_occ=q[K * K:] < ALPHA,
                      p_T=p[:K * K].reshape(K, K))
    return out

def diagram(ax, g, ref, edges, wmax, omax, colour):
    """One group's state diagram; `ref` (Normal) adds the ring and the highlights."""
    ang = np.pi / 2 - 2 * np.pi * np.arange(K) / K
    pos = np.column_stack([np.cos(ang), np.sin(ang)])
    radius = lambda o: .05 + .17 * np.sqrt(o / omax)                    # area ~ time in state
    T, accent = g["T"], ref is not None
    nodes = []
    for k in range(K):
        hit_occ = accent and g["sig_occ"][k]
        nodes.append(ax.add_patch(Circle(pos[k], radius(g["occ"][k]), zorder=4, lw=1.2 if hit_occ else .7,
                                         fc=matplotlib.colors.to_rgba(colour, .35) if hit_occ else NODE,
                                         ec=colour if hit_occ else INK2)))
    for i, j in edges:                                                  # transitions
        hit = accent and g["sig_T"][i, j]
        a = FancyArrowPatch(pos[i], pos[j], connectionstyle="arc3,rad=.18", arrowstyle="-|>",
                            patchA=nodes[i], patchB=nodes[j], shrinkA=2, shrinkB=3,
                            mutation_scale=6 + 6 * T[i, j] / wmax, lw=.3 + 4.5 * T[i, j] / wmax,
                            color=colour if hit else (INK2 if not accent else GREY),
                            alpha=1 if hit else .7, zorder=3 if hit else 2)
        ax.add_patch(a)
        if hit:                                                         # label at the curve's middle
            (x1, y1), (x2, y2) = pos[i], pos[j]
            mx, my = (x1 + x2) / 2 + .09 * (y2 - y1), (y1 + y2) / 2 - .09 * (x2 - x1)
            ax.text(mx, my, f"{100 * (T[i, j] - ref['T'][i, j]):+.0f} pp", fontsize=6.5, color=colour,
                    ha="center", va="center", zorder=5,
                    bbox=dict(boxstyle="round,pad=.15", fc=SURFACE, ec="none", alpha=.85))
    for k in range(K):                                                  # loops, rings, labels
        u = pos[k] / np.linalg.norm(pos[k])
        r = radius(g["occ"][k])
        hit_stay = accent and g["sig_T"][k, k]
        th = np.degrees(np.arctan2(u[1], u[0]))
        ax.add_patch(Arc(pos[k] + u * (r + .045), .11, .11, theta1=th - 140, theta2=th + 140,
                         lw=.3 + 3 * T[k, k], color=colour if hit_stay else (INK2 if not accent else GREY),
                         zorder=2))
        hit_occ = accent and g["sig_occ"][k]
        if accent:                                                      # the Normal group's size
            ax.add_patch(Circle(pos[k], radius(ref["occ"][k]), fc="none", ec=INK2, lw=.7, ls=(0, (2, 2)), zorder=5))
        ax.text(*pos[k], str(k), ha="center", va="center", fontsize=7.5, color=INK, zorder=6)
        lab = f"{100 * g['occ'][k]:.0f}%"
        if hit_occ:
            lab += f" ({100 * (g['occ'][k] - ref['occ'][k]):+.0f} pp)"
        if hit_stay:
            lab += f"\nstay {100 * (T[k, k] - ref['T'][k, k]):+.0f} pp"
        out = pos[k] * (1 + (r + .3) / np.linalg.norm(pos[k]))
        ax.text(*out, lab, ha="center", va="center", fontsize=6.5,
                color=colour if (hit_occ or hit_stay) else INK2, zorder=6)
    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.55, 1.55)
    ax.set_aspect("equal")
    ax.axis("off")

fig, axes = plt.subplots(len(TASKS), len(GROUPS), squeeze=False, figsize=(4.6 * len(GROUPS), 4.7 * len(TASKS)))
dyn3 = {}
for i, task in enumerate(TASKS):
    G = dyn3[task] = groups_of(task)
    N = G["Normal"]
    off = [(a, b) for a in range(K) for b in range(K) if a != b]
    edges = set(sorted(off, key=lambda e: -N["T"][e])[:EDGE_TOP])
    edges |= {e for g in TARGETS for e in off if G[g]["sig_T"][e]}         # every change is drawn
    wmax = max(G[g]["T"][e] for g in GROUPS for e in edges)
    omax = max(G[g]["occ"].max() for g in GROUPS)
    for j, (g, colour) in enumerate(GROUPS.items()):
        ref = None if g == "Normal" else N
        diagram(axes[i, j], G[g], ref, sorted(edges), wmax, omax, colour)
        n_hits = 0 if ref is None else int(G[g]["sig_T"].sum() + G[g]["sig_occ"].sum())
        sub = "" if ref is None else f"  ·  {n_hits} change(s) vs Normal"
        axes[i, j].set_title(f"{task} · {g if ref is None else g + ', ' + CASE_FROM + '+'} (n={G[g]['n']}){sub}",
                             fontsize=9.5, color=INK if ref is None else colour)
fig.legend(handles=[
    Line2D([], [], ls="none", marker="o", ms=10, mfc=NODE, mec=INK2, label="circle area = time in state (number = %)"),
    Line2D([], [], color=INK2, lw=2.5, label="arrow width = p(j | i), the share of moves out of i that go to j"),
    Line2D([], [], ls="none", marker=r"$\circlearrowleft$", ms=12, color=INK2, label="loop = p(stay): thicker = held longer"),
    Line2D([], [], ls="none", marker="o", ms=10, mfc="none", mec=INK2, label="dashed ring = the Normal group's time in state"),
    *[Line2D([], [], color=c, lw=2.5, label=f"{g}: differs from Normal (FDR<{ALPHA:g}, adjusted); label = change in pp")
      for g, c in GROUPS.items() if g != "Normal"]],
    loc="lower center", ncol=3, fontsize=7.5, bbox_to_anchor=(.5, -.005))
fig.suptitle(f"3  state dynamics · {set_label(pick)}", x=.01, ha="left", fontsize=10)
fig.tight_layout(rect=(0, .07, 1, .97))
save(fig, "fig3_state_diagrams")

In [ ]:
# Every change vs Normal that survives FDR: transitions, stays and occupancy, in percentage points.
rows = []
for task, G in dyn3.items():
    for g in TARGETS:
        for a, b in zip(*np.nonzero(G[g]["sig_T"])):
            what = f"stay in {a}" if a == b else f"{a} -> {b}"
            rows.append((task, g, what, G["Normal"]["T"][a, b], G[g]["T"][a, b], G[g]["p_T"][a, b]))
        for k in np.nonzero(G[g]["sig_occ"])[0]:
            rows.append((task, g, f"time in {k}", G["Normal"]["occ"][k], G[g]["occ"][k], np.nan))
pd.DataFrame(rows, columns=["task", "group", "what", "normal", "group_value", "p"]).assign(
    change_pp=lambda t: 100 * (t["group_value"] - t["normal"])).round(4)